# PodcastIQ — AI Startup Podcast Studio
*Automated two-host podcast generator powered by Claude + gTTS*

## Step 1 — Install Dependencies

In [ ]:
!pip install -r ../requirements.txt

## Step 2 — Load Environment Variables

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load API keys from .env
load_dotenv(Path("../.env"))

# Add src/ to path so we can import our modules
sys.path.insert(0, str(Path("../src").resolve()))

print("✓ Environment loaded")
print("✓ ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Step 3 — Import Pipeline Modules

In [ ]:
from models import PodcastSettings, PodcastStyle, SourceType
from data_processor import process
from llm_processor import generate_script
from tts_generator import synthesise_script

print("✓ All modules imported successfully")

## Step 4 — Quick Pipeline Test (Text Input)
Test the full pipeline with a short text before launching the UI.

In [ ]:
test_text = """
Anthropic has raised $2.75 billion in its latest funding round, valuing the AI startup at $18.4 billion.
The company, founded by former OpenAI researchers, develops Claude — a family of large language models
designed with a focus on safety and reliability. The funding will be used to scale infrastructure,
expand research teams, and compete with OpenAI's GPT-4o and Google's Gemini models.
Investors include Google, Spark Capital, and a range of sovereign wealth funds.
Analysts say the raise signals continued investor confidence in foundation model companies despite
rising compute costs and increasing competition from open-source alternatives like Meta's Llama 3.
"""

settings = PodcastSettings(
    style=PodcastStyle.EDUCATIONAL,
    host_a_name="Alex",
    host_b_name="Sam",
    target_minutes=3,
)

# Step 1: Process input
podcast_input = process(test_text, SourceType.TEXT)
print(f"✓ Input processed: {podcast_input.word_count} words")

# Step 2: Generate script
print("Generating script with Claude... (this takes ~10 seconds)")
script = generate_script(podcast_input, settings)
print(f"✓ Script generated: {len(script.lines)} dialogue lines")
print(f"  Title: {script.metadata.title}")
print(f"  Tags: {', '.join(script.metadata.tags)}")

# Step 3: Preview script
print("\n--- SCRIPT PREVIEW ---")
for line in script.lines[:6]:
    print(f"{line.host_name.upper()}: {line.text[:80]}...")

## Step 5 — Generate Audio
Once the script looks good, convert it to audio.

In [ ]:
print("Generating audio... (this takes ~30 seconds depending on episode length)")
audio = synthesise_script(script, settings)
print(f"✓ Audio saved to: {audio.file_path}")
print(f"  Duration: {round(audio.duration_seconds / 60, 1)} minutes")

# Play audio inline in the notebook
from IPython.display import Audio
Audio(audio.file_path)

## Step 6 — Launch the Gradio UI
Full interactive interface with all input types and settings.

In [ ]:
import subprocess
subprocess.Popen(["python", str(Path("../src/main.py").resolve())])

print("Gradio app launching at http://localhost:7860")